In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess
import shutil
import sys
import os
import re


# ============================================================
# 설정
# ============================================================

DRIVE_PROJECT = Path(
    "/content/drive/MyDrive/Colab Notebooks/비교/16k/qwen3_vl_mmmu_eval/qwen3_vl_mmmu_eval"
)

LOCAL_PROJECT = Path("/content/qwen3_vl_mmmu_eval")

DATA_ROOT = Path("/content/drive/MyDrive/mmmu_eval/data")

# False = 과목당 1문항 Smoke Test
# True  = 전체 900문항
RUN_FULL = True

cfg_path = DRIVE_PROJECT / "configs" / "mmmu_eval.yaml"
text = cfg_path.read_text(encoding="utf-8")

# 숫자에 붙어 있는 # 앞에 공백 추가
text = re.sub(r"(?<=\d)#", " #", text)

cfg_path.write_text(text, encoding="utf-8")
print("수정 완료:", cfg_path)


# ============================================================
# 1. Google Drive 연결
# ============================================================

drive.mount("/content/drive")

print("\n[1/6] Google Drive 연결 완료")


# ============================================================
# 2. 프로젝트 확인
# ============================================================

assert DRIVE_PROJECT.exists(), (
    f"프로젝트 폴더가 없습니다:\n{DRIVE_PROJECT}"
)

assert (DRIVE_PROJECT / "requirements.txt").exists(), (
    "requirements.txt 파일이 없습니다."
)

assert (DRIVE_PROJECT / "scripts/run_mmmu_eval.sh").exists(), (
    "scripts/run_mmmu_eval.sh 파일이 없습니다."
)

assert DATA_ROOT.exists(), (
    f"MMMU 데이터 폴더가 없습니다:\n{DATA_ROOT}"
)

print("[2/6] 프로젝트 및 데이터 확인 완료")


# ============================================================
# 3. 프로젝트를 Colab 로컬 /content로 복사
# ============================================================

if LOCAL_PROJECT.exists():
    shutil.rmtree(LOCAL_PROJECT)

shutil.copytree(
    DRIVE_PROJECT,
    LOCAL_PROJECT,
    ignore=shutil.ignore_patterns(
        ".venv",
        "__pycache__",
        "*.pyc"
    )
)

print("[3/6] 프로젝트 복사 완료")
print(LOCAL_PROJECT)


# ============================================================
# 4. requirements 설치
# ============================================================

print("\n[4/6] 패키지 설치 시작\n")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(LOCAL_PROJECT / "requirements.txt")
    ],
    check=True
)

print("\n✅ 패키지 설치 완료")


# ============================================================
# 5. pytest
# ============================================================

# print("\n[5/6] 프로젝트 테스트 실행\n")

# subprocess.run(
#     [
#         sys.executable,
#         "-m",
#         "pytest",
#         "tests/",
#         "-q"
#     ],
#     cwd=str(LOCAL_PROJECT),
#     check=True
# )

# print("\n✅ pytest 통과")


# ============================================================
# GPU 확인
# ============================================================

print("\nGPU 확인\n")

subprocess.run(
    ["nvidia-smi"],
    check=True
)


# ============================================================
# 6. MMMU 실행
# ============================================================

env = os.environ.copy()

# 가상환경 대신 현재 Colab Python 사용
env["PYTHON_BIN"] = sys.executable

print("\n현재 사용 Python:")
print(sys.executable)


# ------------------------------------------------------------
# 전체 900문항
# ------------------------------------------------------------

if RUN_FULL:

    OUTPUT_DIR = (
        "/content/drive/MyDrive/mmmu_eval/outputs/baseline_seed42_16k"
    )

    cmd = [
        "bash",
        "scripts/run_mmmu_eval.sh",

        "--model_path",
        "Qwen/Qwen3-VL-4B-Instruct",

        "--model_revision",
        "ebb281ec70b05090aa6165b016eac8ec08e71b17",

        "--data_root",
        str(DATA_ROOT),

        "--config",
        "configs/mmmu_eval.yaml",

        "--seed",
        "42",

        "--output_dir",
        OUTPUT_DIR,

        "--require_gpu",
        "A100",

        "--stage",
        "all"
    ]

    print("\n[6/6] 🚀 900문항 전체 평가 시작")


# ------------------------------------------------------------
# Smoke Test
# ------------------------------------------------------------

else:

    OUTPUT_DIR = (
        "/content/drive/MyDrive/mmmu_eval/outputs/smoke_seed42_full"
    )

    cmd = [
        "bash",
        "scripts/run_mmmu_eval.sh",

        "--model_path",
        "Qwen/Qwen3-VL-4B-Instruct",

        "--model_revision",
        "ebb281ec70b05090aa6165b016eac8ec08e71b17",

        "--data_root",
        str(DATA_ROOT),

        "--config",
        "configs/mmmu_eval.yaml",

        "--seed",
        "42",

        "--output_dir",
        OUTPUT_DIR,

        "--require_gpu",
        "A100",

        "--limit_per_subject",
        "1",

        "--stage",
        "all"
    ]

    print("\n[6/6] 🧪 Smoke Test 시작")


# subprocess.run(
#     cmd,
#     cwd=str(LOCAL_PROJECT),
#     env=env,
#     check=True
# )

env["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    cmd,
    cwd=str(LOCAL_PROJECT),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
proc.wait()

if proc.returncode != 0:
    raise RuntimeError(f"❌ 스크립트 실패 (exit code {proc.returncode}) — 위 로그의 마지막 Traceback 확인")

print("\n" + "=" * 60)
print("✅ 실행 완료")
print("=" * 60)
print("결과 폴더:")
print(OUTPUT_DIR)
print("=" * 60)

수정 완료: /content/drive/MyDrive/Colab Notebooks/비교/16k/qwen3_vl_mmmu_eval/qwen3_vl_mmmu_eval/configs/mmmu_eval.yaml
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

[1/6] Google Drive 연결 완료
[2/6] 프로젝트 및 데이터 확인 완료
[3/6] 프로젝트 복사 완료
/content/qwen3_vl_mmmu_eval

[4/6] 패키지 설치 시작


✅ 패키지 설치 완료

GPU 확인


현재 사용 Python:
/usr/bin/python3

[6/6] 🚀 900문항 전체 평가 시작
GPU: NVIDIA A100-SXM4-40GB
2026-09-26 08:50:53.114123: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-26 08:50:53.185247: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_V

한 번에 결과 확인

In [ ]:
import json, collections
from pathlib import Path

out = Path("/content/drive/MyDrive/mmmu_eval/outputs/baseline_seed42_16k")

print("=== 파일 목록 ===")
for p in sorted(out.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(out)}  ({p.stat().st_size:,} bytes)")

print("\n=== raw_outputs.jsonl ===")
rows = [json.loads(l) for l in open(out / "raw_outputs.jsonl")]
ids = [r["id"] for r in rows]
print("총 문항:", len(rows), "| 과목 수:", len({r["subject"] for r in rows}),
      "| 중복 ID:", len(ids) - len(set(ids)))
print("finish_reason:", dict(collections.Counter(r["finish_reason"] for r in rows)))
print("빈 응답:", sum(not r["response"].strip() for r in rows))
print("입력 토큰 최대:", max(r["input_tokens"] for r in rows),
      "| 출력 토큰 최대:", max(r["output_tokens"] for r in rows))

print("\n=== 샘플 응답 3개 ===")
for r in rows[:3]:
    print(f"[{r['id']}] 정답={r['answer']}")
    print("  ", r["response"][-300:].replace("\n", " "), "\n")

print("=== 메타 JSON ===")
for p in out.glob("*.json"):
    try:
        d = json.load(open(p))
    except Exception:
        continue
    if isinstance(d, dict):
        keys = ["status", "model_revision", "seed", "gpu", "evaluated_total", "generation_seconds"]
        print(p.name, {k: d.get(k) for k in keys if k in d})
        for k in ("accuracy", "overall_accuracy", "overall", "scores"):
            if k in d:
                print("  ", k, "=", d[k])

vram = out / "vram_samples.jsonl"
if vram.exists():
    lines = vram.read_text().splitlines()
    print("\n=== VRAM 샘플 (마지막 1개) ===\n", lines[-1] if lines else "없음")

=== 파일 목록 ===
.run.lock  (0 bytes)
chat_template.jinja  (5,292 bytes)
input_lengths.json  (218,523 bytes)
overflow_report.json  (3 bytes)
raw_outputs.jsonl  (9,647,948 bytes)
resolved_config.json  (1,992 bytes)
results_table.md  (1,547 bytes)
run_meta.json  (41,847 bytes)
scored.jsonl  (9,873,254 bytes)
scores.json  (5,610 bytes)
vram_samples.jsonl  (87,406 bytes)

=== raw_outputs.jsonl ===
총 문항: 900 | 과목 수: 30 | 중복 ID: 0
finish_reason: {'length': 92, 'stop': 808}
빈 응답: 0
입력 토큰 최대: 6700 | 출력 토큰 최대: 16384

=== 샘플 응답 3개 ===
[validation_Accounting_1] 정답=B
   turing overhead**, and they are using the average cost per unit for 25,000 units — which is 3+6+2 = $11 — not in options.  I think I have to choose the closest option — but none are close.  Wait — perhaps the question is asking for ** the per unit cost of manufacturing overhead**, and they are using 

[validation_Accounting_2] 정답=C
   , and dividends reduce it.  So:  61,626 = 24,000 + 31,315 + 18,311 - D  Compute right-hand side:  24,

In [ ]:
import json
from pathlib import Path

out = Path("/content/drive/MyDrive/mmmu_eval/outputs/baseline_seed42_16k")
print((out / "results_table.md").read_text())

rows = [json.loads(l) for l in open(out / "scored.jsonl")]
print("scored.jsonl 키:", list(rows[0].keys()))

key = next(k for k in ("correct", "is_correct", "score", "match") if k in rows[0])
def acc(rs):
    return sum(bool(r[key]) for r in rs) / len(rs) * 100 if rs else 0

stop = [r for r in rows if r["finish_reason"] == "stop"]
cut  = [r for r in rows if r["finish_reason"] == "length"]
print(f"\n전체 정확도: {acc(rows):.1f}%  (n={len(rows)})")
print(f"정상 종료(stop): {acc(stop):.1f}%  (n={len(stop)})")
print(f"잘림(length):    {acc(cut):.1f}%  (n={len(cut)})")

# 실제 적용된 설정 확인
cfg = json.load(open(out / "resolved_config.json"))
print("\nmax_new_tokens:", cfg.get("max_new_tokens"), "| max_model_len:", cfg.get("max_model_len"))

| No. | Subject | Data Num | Acc |
|---|---|---|---|
| 1 | Accounting | 30 | 70.00 |
| 2 | Agriculture | 30 | 50.00 |
| 3 | Architecture_and_Engineering | 30 | 46.67 |
| 4 | Art | 30 | 43.33 |
| 5 | Art_Theory | 30 | 60.00 |
| 6 | Basic_Medical_Science | 30 | 66.67 |
| 7 | Biology | 30 | 40.00 |
| 8 | Chemistry | 30 | 36.67 |
| 9 | Clinical_Medicine | 30 | 30.00 |
| 10 | Computer_Science | 30 | 50.00 |
| 11 | Design | 30 | 50.00 |
| 12 | Diagnostics_and_Laboratory_Medicine | 30 | 10.00 |
| 13 | Economics | 30 | 70.00 |
| 14 | Electronics | 30 | 50.00 |
| 15 | Energy_and_Power | 30 | 33.33 |
| 16 | Finance | 30 | 60.00 |
| 17 | Geography | 30 | 33.33 |
| 18 | History | 30 | 40.00 |
| 19 | Literature | 30 | 43.33 |
| 20 | Manage | 30 | 50.00 |
| 21 | Marketing | 30 | 86.67 |
| 22 | Materials | 30 | 46.67 |
| 23 | Math | 30 | 43.33 |
| 24 | Mechanical_Engineering | 30 | 40.00 |
| 25 | Music | 30 | 33.33 |
| 26 | Pharmacy | 30 | 60.00 |
| 27 | Physics | 30 | 56.67 |
| 28 | Psychology | 30 